[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafaellopesdesa/ML_Mandi/blob/main/activities/application/Exercise_2_1_visualize_data.ipynb)

# Exercise 2.1 — Visualise the data

HSF-India lectures at IIT Mandi. Adapted from the [ML4HEP-TIFR NSBI tutorial](https://github.com/iris-hep/nsbi-lhc-toolkit/tree/b1d3cb8c704ee7a54c99367cf6578e5244ac2671/workshops/ml4hep_tifr_colab).

Run the setup cell first, then run the cells in order. Use a **CPU runtime**.
Everything is generated on the runtime's local disk: **20,000 signal and 20,000 background events**, with seed 42.
Training uses two hidden layers of 64 neurons and at most 20 epochs. The expected physics yields are unchanged, so the smaller simulated samples give noisier estimates.
The original exercises and worked likelihood implementations are retained.

Local files last for the lifetime of the runtime. See [the activity guide](https://github.com/rafaellopesdesa/ML_Mandi/tree/main/activities/application) for the notebook order and optional model handoff.


In [ ]:
# Run first: local storage only, in Colab or local Jupyter.
import os
import sys
from pathlib import Path
from urllib.request import urlretrieve

REPOSITORY = "rafaellopesdesa/ML_Mandi"
REF = "main"
N_BKG = N_SIG = 20_000
SEED = 42

if "google.colab" in sys.modules:
    WORKDIR = Path("/content/ML_Mandi_application")
elif (Path.cwd() / "setup_local.py").is_file():
    WORKDIR = Path.cwd()
elif (Path.cwd() / "activities/application/setup_local.py").is_file():
    WORKDIR = Path.cwd() / "activities/application"
else:
    WORKDIR = Path.cwd() / "ML_Mandi_application"

WORKDIR = WORKDIR.resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
helper = WORKDIR / "setup_local.py"
if not helper.is_file():
    urlretrieve(
        f"https://raw.githubusercontent.com/{REPOSITORY}/{REF}/activities/application/setup_local.py",
        helper,
    )
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

from setup_local import setup
setup(WORKDIR, repository=REPOSITORY, ref=REF, n_bkg=N_BKG, n_sig=N_SIG, seed=SEED)


In [ ]:
import pandas as pd
import numpy as np
import mplhep as hep
import matplotlib.pyplot as plt
hep.style.use("ATLAS")

In [ ]:
BASE_PATH = "./dataframes/"

In [ ]:
signal = pd.read_parquet(f"{BASE_PATH}/signal.parquet")
background = pd.read_parquet(f"{BASE_PATH}/background.parquet")

### Visualising the dataset

Before training the NN, lets take a look at what the "kinematics" of the final state look like.

Note how some variables are more discriminating than others, and how several of them are **multi-modal** (more than one bump). Our goal with SBI is to use all of the variables — including their high-dimensional correlations — to build the model.


In [ ]:
variables = ["x1", "x2", "x3", "x4", "x5"]
truth = ["z1", "z2", "z3", "z4", "z5"]

mosaic = [
    ["x1", "x1", "x2", "x2"],
    ["x3", "x3", "x4", "x4"],
    [".",  "x5", "x5", "."],
]
fig, axes = plt.subplot_mosaic(mosaic, figsize=(12, 14))

for var,tr in zip(variables,truth):
    ax = axes[var]

    x_sig = np.array(signal[var])
    x_sig_tr = np.array(signal[tr])
    h_sig, bins = np.histogram(x_sig, bins=50)
    h_sig_tr, bins = np.histogram(x_sig_tr, bins=50)
    hep.histplot(h_sig, bins, ax=ax, label="signal", color='blue', linewidth=1.0)
    hep.histplot(h_sig_tr, bins, ax=ax, label="signal truth", ls='--', color='blue', linewidth=1.0)

    x_bkg = np.array(background[var])
    x_bkg_tr = np.array(background[tr])
    h_bkg, bins = np.histogram(x_bkg, bins=50)
    h_bkg_tr, bins = np.histogram(x_bkg_tr, bins=50)
    hep.histplot(h_bkg, bins, ax=ax, label="background", linewidth=1.0, color="green")
    hep.histplot(h_bkg_tr, bins, ax=ax, label="background truth", ls='--', linewidth=1.0, color="green")

    ax.legend(fontsize=14)
    ax.set_xlabel(var, size=16)
    ax.set_ylabel("Events", size=16)

fig.tight_layout()